In [ ]:
import json
from typing import Dict, List

file_path = r"..\data\crawler\weibo_trending\weibo_trending_data.json"
def read_data(file_path: str) -> List[Dict]:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

trendings = read_data(file_path)


len(trendings), trendings[:5]

(157437,
 [{'date': '2025-01-01', 'name': '赵露思发长文回应', 'type': '暂无', 'clicks': 14913080},
  {'date': '2025-01-01', 'name': '新年快乐', 'type': '其他', 'clicks': 10647970},
  {'date': '2025-01-01', 'name': '种地吧直播', 'type': '暂无', 'clicks': 3189152},
  {'date': '2025-01-01',
   'name': '总书记的这些话暖心鼓劲',
   'type': '社会',
   'clicks': 2512950},
  {'date': '2025-01-01', 'name': '银河酷娱致歉', 'type': '明星', 'clicks': 2425505}])

In [52]:
from collections import Counter

type_counter = Counter(item["type"].strip() for item in trendings
                       if len(item["type"]) < 10)

print("词条数最多的前20个类型：")
for type_name, count in type_counter.most_common(20):
    print(f"  {type_name}: {count:,} 条")

词条数最多的前20个类型：
  社会: 48,700 条
  暂无: 44,536 条
  明星: 13,936 条
  体育: 8,185 条
  明星-内地: 4,836 条
  时事: 4,680 条
  游戏: 4,240 条
  电视剧: 3,472 条
  财经: 3,219 条
  综艺: 2,599 条
  电视剧-国产剧: 1,701 条
  搞笑: 1,540 条
  综艺-内地综艺: 1,319 条
  电影: 1,298 条
  汽车: 1,282 条
  互联网: 865 条
  其他: 813 条
  音乐: 691 条
  科普: 675 条
  情感: 653 条


In [53]:
import random

selected_types = ["社会", "时事", "财经", "互联网", "科普", "情感"]

selected_trendings = [item for item in trendings if item["type"] in selected_types]

print(f"筛选后词条数: {len(selected_trendings)}")
print(f"占比: {len(selected_trendings) / len(trendings) * 100:.1f}%")
print("筛选词条示例：")

for item in random.sample(selected_trendings, 5):
    print(f" {item['date']} - {item['name']} - [{item['type']}] - {item['clicks']:,} 次")

筛选后词条数: 58792
占比: 37.3%
筛选词条示例：
 2025-01-19 - 多款手机价格集体降至6000元以内 - [社会] - 435,739 次
 2025-03-15 - 我国各区域外贸开年成绩单出炉 - [社会] - 908,466 次
 2025-12-13 - 日军南京杀人竞赛有人杀了106人 - [时事] - 225,995 次
 2025-04-24 - 妻子开车躲家暴致丈夫身亡被判11年 - [社会] - 689,231 次
 2025-03-11 - 95后女教师勇救轻生男孩 - [社会] - 235,235 次


In [54]:
import numpy as np

# 提取所有点击量
clicks = [item["clicks"] for item in selected_trendings]

ratio = 99

threshold = np.percentile(clicks, ratio)

print(f"总词条数: {len(selected_trendings)}")
print(f"点击量{ratio}%分位数: {threshold:,.0f}")
print(f"点击量范围: {min(clicks):,.0f} - {max(clicks):,.0f}")
# print(f"平均点击量: {np.mean(clicks):,.0f}")
# print(f"中位数点击量: {np.median(clicks):,.0f}")

top_1_percent_trendings = [item for item in selected_trendings 
                            if item["clicks"] >= threshold]

# 按点击量降序排序
top_1_percent_trendings.sort(key=lambda x: x["clicks"], reverse=True)

print(f"\n筛选后词条数: {len(top_1_percent_trendings)}")
print(f"占比: {len(top_1_percent_trendings) / len(trendings) * 100:.1f}%")

总词条数: 58792
点击量99%分位数: 1,991,206
点击量范围: 0 - 25,245,679

筛选后词条数: 588
占比: 0.4%


In [55]:
print("热搜词条示例：")
for item in random.sample(top_1_percent_trendings, 5):
    print(f"{item['date']} - {item['name']} - [{item['type']}] - {item['clicks']:,} 次")


热搜词条示例：
2025-04-10 - 美国被加关税后特朗普呼吁冷静 - [时事] - 7,605,554 次
2025-08-15 - 董某莹成绩单造假 - [社会] - 9,610,327 次
2025-09-23 - 桦加沙登陆地点确认 - [社会] - 2,052,809 次
2025-07-27 - 数智新时代电商新价值 - [社会] - 2,254,021 次
2025-01-07 - 王星弟弟已与其通视频电话 - [社会] - 10,831,049 次


In [56]:
# 查看筛选后的数据统计
print("=" * 60)
print("筛选后数据统计")
print("=" * 60)
print(f"总词条数: {len(top_1_percent_trendings):,}")
print(f"点击量阈值: {threshold:,.0f} 次")
print(f"最高点击量: {top_1_percent_trendings[0]['clicks']:,} 次")
print(f"最低点击量: {top_1_percent_trendings[-1]['clicks']:,} 次")
print(f"平均点击量: {np.mean([item['clicks'] for item in top_1_percent_trendings]):,.0f} 次")

# 按类型统计
from collections import Counter
type_dist = Counter(item['type'] for item in top_1_percent_trendings)
print(f"\n类型分布:")
for type_name, count in type_dist.most_common():
    percentage = count / len(top_1_percent_trendings) * 100
    print(f"  {type_name}: {count:,} 条 ({percentage:.1f}%)")

筛选后数据统计
总词条数: 588
点击量阈值: 1,991,206 次
最高点击量: 25,245,679 次
最低点击量: 1,991,504 次
平均点击量: 4,394,284 次

类型分布:
  社会: 503 条 (85.5%)
  时事: 39 条 (6.6%)
  财经: 31 条 (5.3%)
  互联网: 12 条 (2.0%)
  情感: 2 条 (0.3%)
  科普: 1 条 (0.2%)


In [57]:
# import json

# with open('top_1_percent_trendings.json', 'w', encoding='utf-8') as f:
#     json.dump(top_1_percent_trendings, f, ensure_ascii=False, indent=2)
# print("已保存到: top_1_percent_trendings.json")

In [58]:
with open("top_1_percent_trendings_detail.txt", 'w', encoding='utf-8') as f:
    for item in top_1_percent_trendings:
        f.write(f"{item['date']} - {item['name']} - [{item['type']}] - {item['clicks']:,}\n")

trending_names = set([item['name'] for item in top_1_percent_trendings])

with open("top_1_percent_trendings.txt", 'w', encoding='utf-8') as f:
    for name in trending_names:
        f.write(f"{name}\n")

In [59]:
keywords = """
江苏确诊一例罕见传染病
结婚离婚登记不再需要户口簿了
315
日本流感病例已超950万人
受阅部队已在长安街列阵
祝1335万高考生大胜归来
新年文案
美国袭击伊朗核设施
胖猫事件90多吨食物被浪费
香港大埔火灾约200人情况未明
董某莹四项证书被撤销
福建震感
SU7爆燃遇难者父亲称小米仍没来联系
浪还在琴没了
太二酸菜鱼为何没人吃了
律师解读吸毒记录封存不是彻底清除
李刚被双开
军事航天部队方队首次亮相
马斯克指责USAID资助研发生物武器
法国最新涉台表态
小鹏报警
2026春节放9天假
微信能自动发消息了
协和医学院董某莹事件最新进展
平价不是蜜雪冰城的万能挡箭牌
台湾当归
我国新一代太空感知星座发布
韩国女学生跳车身亡八旬出租车司机无罪
4月国民经济顶住压力稳定增长
西贝道歉
香港大火已致94人遇难
肖飞回应手术过程中离场是吃降压药
一条视频科普流感高发期知识
香港大埔火灾已致128人遇难
向太称刘晓庆被李小冉骂了10分钟
黄杨某甜有关问题最新通报
红旗见证时代荣光
金价大跳水了
辽阳一饭店火灾22死3伤
特朗普突然下令关灯放电视
王星弟弟已与其通视频电话
斯凯奇宣布退市
流感为何会致命
美国一边打贸易战一边求鸡蛋
武大撤销肖某某记过处分
七彩油菜花美得像莫奈花园
1斤虾仁7两冰
河南出现15级大风
茉莉奶白被砸身亡店员系17岁男生
官方通报江秋莲被举报诈捐
日本全面取消女警裙装制服
台湾台东5.8级地震
这样的西藏你见过吗
余华英死刑后杨妞花回应是否会退网
殡仪馆证实网红王子墨去世
母子看唐探强行占座致300多人退票
93年女孩成市值407亿公司董事长
女儿调座椅压死儿子家长索赔车企
黄晓明称朱媛媛在剧组都没有说过患癌
新郎去世新娘终止妊娠被判还彩礼
警方通报受胡雷资助女孩去世
韩某某投敌叛变48小时内被抓
贩毒女头目年仅17岁
台政客与高市早苗办公桌合影被群嘲
广州解除五停
主播直播擦边卖枕头2小时场观30万
九三阅兵具体安排来了
清华大学今晚给庞众望颁奖
套圈中170万玛莎拉蒂男子称想折现
借贷宝
爷爷为夺回孙子被人贩踩出眼珠
董某莹成绩单造假
从古诗词中感受立春之美
什么是爆肺
央视曝光美甲灯安全隐患
公司错发1.65亿工资员工被判不退款
青岛大学凌晨发情况说明
女孩凌晨店内熟睡突遭陌生男子闯入
日本学者点破特朗普真实想法
香港火灾已有75人遇难
中美日内瓦经贸会谈联合声明
蔡国强致歉
美国输华商品已无市场接受可能性
2026年这么干
十三州府取关曾黎
这3个带马成语太给劲了
升至125%后为何中国不陪了
协和往年4加4录取名单已无法查询
苏超泰州队冠军
蔡澜去世
结婚前一天新郎新娘先后坠亡
中美相互24%关税90天内暂停实施
何卫东等9人被开除党籍军籍
哪吒2被删减画面首度公开
赵露思艾特银河酷娱
521
法院向吴亦凡经纪公司追缴诉讼费
湖南卫健委回应罗帅宇事件
美媒称145%关税无异于对华贸易禁运
中国台湾省
官方通报余杭自来水异味
最新版全国入冬进程图
教皇方济各去世
中国被曝已不再生产出口美国的玩具
多位演员自曝曾被导演郑某某侵害
正月初六把穷气送出门
公安机关公布王星被骗至缅甸细节
春联中藏着中国年独有的浪漫与美好
每个中国人传唱的精神密码
3名中国公民在塔遇袭身亡
放飞和平鸽
王星成功获救
电子签放款人竟不是活人
泰国宣布禁娱30天
哪吒2感谢140000000位观众
中方回应特朗普威胁额外征收50%关税
饺子导演已闭关
致敬暴雨中向险而行的身影
7月这些新规影响你我
哪吒2突破99亿
太原老葛被罚560万元
泰国喊话中国游客
警方通报嘎子哥行拘7天
全球股市巨震
湘雅二医院两年间给罗帅宇转账40多万
法工委回应吸毒记录封存相关问题
民乐百鸟朝凤的控场能力
未来五年什么工作吃香
女生一觉醒来相亲对象已站床边
艺人吃中国饭砸中国锅绝不容忍
顾茜茜抖音账号被永久封禁
世纪婴儿去世鉴定结果已出
315晚会
肖庆平车祸离世
翻新卫生巾
休2上4休1
王暖暖国内离婚要等到70岁
蜜雪冰城早餐仅在部分城市试点
汪小菲说大S是我的家人
一张截图钓出多少反华党
云南通报赤裸小孩事件
香港起火大楼内发现生还者
韩国媒体称李在明当选韩国总统
香港火灾已致44人遇难
雷军发长文
大年初一票房排名
中国赴美游客风险提示
香港将下半旗志哀
Zara广告因模特太瘦被禁
日本流感到底有多严重
数智新时代电商新价值
打卡夏天里的大美中国
金正恩抵达
哀牢山发现7株冥界之花
特朗普遭死亡威胁
6名学生内蒙古遇难最新细节披露
尼斯湖水怪再次被拍到
马拉松一对男女亲密动作被拍到
东风5C打击范围覆盖全球
猫一杯案开庭
中国人从太空发回拜年视频
湖大失联女生遗体已找到
郭富城妻子称在米兰被抢劫
泰国DJ与毒枭老大女友有染遭枪杀
武大校长回应是否撤销肖同学处分
高考首日旗开得胜
陈梦晒年夜饭
今年春运购票2个关键时间点
肖飞被开除董袭莹问题何时查
市监总局约谈饿了么美团京东
设台湾光复纪念日
虐猫考生被取消招聘资格
女装退货率
上海老人离世430万和1套房无人继承
月全食
27岁女游客在三亚被蛇咬伤身亡
央视还原胖猫事件真相
涉事医院回应小洛熙尸检报告
内蒙古通报那尔那茜有关核查情况
女生高考涨了167分被吓到模糊
欢喜哥许绍雄逝世
五金市场公然售卖非标电线电缆
3岁以下婴幼儿每年补贴3600元
被车撞了6天后突然去世该谁担责
天水一幼儿园幼儿血铅异常8人被刑拘
血铅事件27人被追责问责
苏菲发声明
6组数据感受中国外贸韧性
悟空和哪吒风靡全球
茉莉奶白被曝喝出完整塑料袋
中方不同意台湾参加今年的世卫大会
特朗普承认观看中国阅兵
婚后出轨生子不等于重婚罪
连休8天的长假来了
日方竟要求中方解释
湾区升明月
日本商家不断收到中国游客取消通知
江苏一地最新公布新生儿爆款名字
张碧晨方说无需多言
被老板强奸女高管老公回应
外交部回应美国对华征收104%关税
卫健委最新通报肖某董某莹事件
雷军回应小米汽车续航测试
南京红老头系38岁男子
女民兵方队亮相
12306回应孕妇被行李箱砸中致早产
崔丽丽穿被性侵时衣服出庭
刘国梁辞职
余杭自来水异味调查情况
东部战区进逼
民警上门要求删视频的监控曝光
日本末世预言剩1天
涨工资的信号越来越明确
台湾艺人带中国台湾省话题发博
细数高市早苗九宗罪
五三惨案
聋哑女生因长相太完美被质疑
尹锡悦被罢免总统职务
女子偷偷用公共餐具喂狗
释永信与多名女性保持不正当关系
日本再现无差别杀人
王楚钦重庆赛夺冠
外媒称美国的痛苦是自作自受
爱达邮轮取消所有日本目的地
余华英被执行死刑
马丽演上沈腾丈母娘了
毛主席嫡孙毛新宇回韶山祭拜爷爷
新修订婚姻登记条例自5月10日起施行
空着的1945检阅车
外交部回应美对华加征245%关税
关税战贸易战中方不愿打但也不怕打
北京交警通报陈震发生车祸
香港火灾已致13死28伤
杭州就娃哈哈事件成立专班
女孩偷拿妈妈百万珠宝卖了60元
英国水煮遗体排入下水道或将合法化
男子转嫖娼对象138万原配追讨败诉
男子诋毁九三阅兵被拘
哈佛大学回应
武大回应图书馆事件
小米王腾因泄密被辞退
4300年前后的石峁城先民找到了
对美所有进口商品加征125%关税
2024年全国结婚登记610.6万对
凤凰传奇天津演唱会取消
警方通报荣梓杉李禹熹纠纷
山姆配送员竟是三逃逃犯
石破茂吃拉面加太多叉烧被痛骂
陶白白官宣离婚
香港火灾救援最新消息
官方辟谣赵露思助农公司获助农大使
日本流感
建议将禁止就业年龄歧视纳入法律
大S
祖冲之二号助力量子模拟新突破
第一名掉准考证的学生出现了
名创优品道歉
奥迪暂停对美国经销商交付新车
杨振宁讣告
知网已搜不到董袭莹论文
李刚被逮捕
电商平台全面取消仅退款
鸿蒙智行称已充分收集证据
总书记的这些话暖心鼓劲
冬至
香港全体公务员冻薪
司马南偷税被罚超900万
罗永浩直播
人民军队时刻准备着
特朗普恢复死刑
易会满涉嫌严重违纪违法
贵州黔西游船侧翻事故约70人落水
陈震偷税案
脆升升薯条道歉
日本胆敢染指台湾就是侵略中国本土
党和人民完全可以信赖解放军
一句句古诗词感受雨水节气
茶百道发文致歉
第一批买哪吒金镯的人已赚麻了
李威妻子称目睹死者受虐全程
小米YU7价格
宗馥莉已经辞职
千亩辣椒免费摘系谣言
和睦家医院回应女明星生产信息疑被泄露
中国经济是大海
西贝承认部分菜品是隔夜菜
星汉耀江城
俄方开停战条件
陈彼得去世
烟雨漓江将中国水墨画具象化了
故宫雪景大片
为啥说小寒胜大寒
靳东两会建议AI换脸立法
刘强东亲自送外卖
男子在生殖医院做手术后次日身亡
长春成功申办2027世界大冬会
比尔盖茨宣布将捐出几乎全部财富
光荣属于劳动者
白象食品道歉
我国海洋工程装备制造连续7年霸榜全球
宗馥莉输了
得流感自救不要错过黄金48小时
广东高质量活力向未来
网友曝小米汽车车主驾驶中睡着
这个踩点变装又燃又帅
雷军喜提1小时首富体验卡
昆明一火车站试验列车撞人致11死
韩红为小洛熙宝宝发声
小米SU7高速碰撞爆燃事件细节
陈震偷税追缴并罚共计247.48万元
深圳全市解除五停
罗大美遗体已在太平间待了752天
少林寺住持释永信被查
TikTok停止在美服务
订婚强奸案二审驳回上诉
9图读懂未来消费新风口
28年五一真能连休9天吗
没人会在35岁突然丧失工作能力
美国声称对华关税加到245%
男生与女友同居太兴奋后空翻摔死
亚洲前首富李兆基逝世
啄木鸟称放弃公关
黄杨钿甜父亲被立案调查
专家称宅基地进入市场价值达1.3万亿
奥斯卡评委喊话饺子导演
哪吒2全球动画第1
警方通报河北一女子疑家暴去世
白象多半桶方便面的多半是商标
排队1小时兑蛇钞蛇币到手就转卖
台湾
小米汽车回应
喜乐安宁中国年
缅甸7.7级左右地震
自由点道歉
2025年高考注意事项
十个勤天春晚第一个节目
南京红姐被抓
90秒感受三夏收获的幸福感
12306回应取消靠窗选座
特朗普关税松口一天就反悔
退休夫妻月入1.2万负债1.2亿
全国人口减少139万人
华为Mate80晨曦金版本已售罄
韩国人被红烧肉羊肉串和火锅硬控
小米发布会
武汉警方通报一地发生伤人事件
6名大学生在企业参观学习时溺亡
男子性侵初中女生致其怀孕不予立案
南京红老头被抓
2025考研国家线发布
护舒宝声明
低于这个价格可能买不到真羽绒
咱家飞机再也不用飞两遍
王楚钦战胜雨果夺冠
香港起火大楼已无大面积明火
王莉霞接受审查调查
香港廉政公署先后拘捕8人
百度副总裁谢广军道歉
香港火灾已开设8个庇护中心
高考288分女生已接到高校联系电话
2000年0时0分出生的世纪婴儿去世
部分日本人开始反对上四休三了
官方通报蔡国强烟花秀
刘国梁回应辞职
韩国人在上海消费能力好强
印巴同意立即停火
一次性内裤爆雷
雅安纪委监委回应黄杨钿甜耳环事件
宁波通报患儿手术后离世
印度坠机事故中发现一名幸存者
全国流感病毒阳性率快速上升
西门子分公司总裁一家在坠机中遇难
女子钻漏洞下单476笔薅羊毛25万
高市再发涉台谬论
金饰价格涨破1000元
外交部回应美对华征收125%关税
演唱会亲密搂抱两人均被停职
阅兵飞机
夏天的凉席是古人严选
以伊宣布正式停火
金昊被判处死刑
李嘉诚要卖43个港口给美国企业
宋慧乔首谈离婚原因
我国再次发射一箭双星
赵露思陷假助农风波
文化中国行过年的仪式感
三只羊东北雨姐再被点名
淘宝免单
印度坠机242人全部遇难
徐大久称演员星星已进入园区
泰媒称演员王星在缅甸被找到
合肥通报三只羊问题调查处置情况
315晚会曝光手机抽奖疯狂敛财
行进看中国
文化中国行二十四节气经典美食
12月这些新规影响你我
铁路春运售票迎高峰
总要去看看今年的冰天雪地吧
健身博主马章浩去世
美因海关系统故障暂未征收关税
ABC拿经期性感当卖点离了大谱
网警护航高考祝学子金榜题名
信息黑洞疯狂窃取个人隐私
阅兵直播
美国被加关税后特朗普呼吁冷静
公积金
iPhone17系列价格
桦加沙登陆地点确认
非遗总数世界第一是什么体验
市监局介入调查赵露思假助农风波
一段录音是订婚强奸案重要证据
315记者为取证吃到吐
人民军队2025年终大片
复婚能否休婚假各省情况不一
王健林再卖48座万达广场
台测试封杀小红书遭网友嘲讽
蛇年怕蛇怎么办
颜十六已到案回国
试验列车撞人事故一伤者意识清醒
杨幂15年前的博文火了
海底捞小便事件10倍现金补偿
韩国空难客机黑匣子撞墙前4分钟停录
央行宣布降准降息
央视揭秘中国核电站内部
清华回应女教授被树砸身亡
何以中国走进福建
王楚钦无缘澳门世界杯决赛
特朗普认怂了
一架波音787客机印度坠毁
爱奇艺道歉
哪吒2爆火保洁3倍工资还要求加钱
A股2025年十大牛股
尹锡悦被逮捕
两千万粉丝小网红母亲回应摆拍质疑
雷军发布小米购置税补贴重要提示
顺丰寄丢价值5万手镯仅赔67元
6000元以下手机补贴最终价格的15%
建议岳云鹏别上春晚
王暖暖成功离婚
泰国考虑对电诈园区断电断网
香港向每户灾民发1万港元补助
女子疑因被拔错牙后坠楼身亡
公办民办幼儿园均可享受免保教费
农村宅基地废弃面积760万公顷
特朗普的关税政策又变了
台球女助教被客人开球砸到手指骨折
高考语文
马龙乒协副主席
西藏定日县地震已致95人遇难
警方捣毁绑架王星案涉案公司
各地端午氛围渐浓
iPhone18Pro或实现FaceID小型化
借台湾生事就是给日本找事
鸡窝头女士收拾漂亮去上班
苹果紧急往美国运iPhone
助理确认蔡磊丧失语言能力
顶级富二代都开始下海拍狗血短剧了
数说我国外贸亮眼成绩单
神22一箭穿日
对美所有进口商品关税再提高50%
2025亚冬会开幕式
猿辅导员工猝死在公司
蛇年新春里的中国
卡塔尔首都多哈发生剧烈爆炸
台湾17岁少年遭好友虐杀弃尸
横店女演员拍戏身亡爆料部分被删
啄木鸟维修
金正恩抵京
"""

keywords = keywords.strip().split('\n')

keywords = [f"#{keyword}#" for keyword in keywords]

keywords = ','.join(keywords)

keywords

'#江苏确诊一例罕见传染病#,#结婚离婚登记不再需要户口簿了#,#315#,#日本流感病例已超950万人#,#受阅部队已在长安街列阵#,#祝1335万高考生大胜归来#,#新年文案#,#美国袭击伊朗核设施#,#胖猫事件90多吨食物被浪费#,#香港大埔火灾约200人情况未明#,#董某莹四项证书被撤销#,#福建震感#,#SU7爆燃遇难者父亲称小米仍没来联系#,#浪还在琴没了#,#太二酸菜鱼为何没人吃了#,#律师解读吸毒记录封存不是彻底清除#,#李刚被双开#,#军事航天部队方队首次亮相#,#马斯克指责USAID资助研发生物武器#,#法国最新涉台表态#,#小鹏报警#,#2026春节放9天假#,#微信能自动发消息了#,#协和医学院董某莹事件最新进展#,#平价不是蜜雪冰城的万能挡箭牌#,#台湾当归#,#我国新一代太空感知星座发布#,#韩国女学生跳车身亡八旬出租车司机无罪#,#4月国民经济顶住压力稳定增长#,#西贝道歉#,#香港大火已致94人遇难#,#肖飞回应手术过程中离场是吃降压药#,#一条视频科普流感高发期知识#,#香港大埔火灾已致128人遇难#,#向太称刘晓庆被李小冉骂了10分钟#,#黄杨某甜有关问题最新通报#,#红旗见证时代荣光#,#金价大跳水了#,#辽阳一饭店火灾22死3伤#,#特朗普突然下令关灯放电视#,#王星弟弟已与其通视频电话#,#斯凯奇宣布退市#,#流感为何会致命#,#美国一边打贸易战一边求鸡蛋#,#武大撤销肖某某记过处分#,#七彩油菜花美得像莫奈花园#,#1斤虾仁7两冰#,#河南出现15级大风#,#茉莉奶白被砸身亡店员系17岁男生#,#官方通报江秋莲被举报诈捐#,#日本全面取消女警裙装制服#,#台湾台东5.8级地震#,#这样的西藏你见过吗#,#余华英死刑后杨妞花回应是否会退网#,#殡仪馆证实网红王子墨去世#,#母子看唐探强行占座致300多人退票#,#93年女孩成市值407亿公司董事长#,#女儿调座椅压死儿子家长索赔车企#,#黄晓明称朱媛媛在剧组都没有说过患癌#,#新郎去世新娘终止妊娠被判还彩礼#,#警方通报受胡雷资助女孩去世#,#韩某某投敌叛变48小时内被抓#,#贩毒女头目年仅17岁#,#台政客与高市早苗办公桌合影被群嘲#,#广州解除五停#,#主播直播擦边卖枕头2小时场观30万#,#九三阅兵具体安排来了#,#清华大学今晚给庞众望颁奖#,#套圈中170万玛莎拉蒂男

In [60]:
import json
from datetime import datetime

with open("search_contents_2026-01-15.json", "r", encoding="utf-8") as f:
    trending_weibos = json.load(f)


def clean_weibos(trending_weibos: List[Dict]) -> List[Dict]:
    target_weibos = []
    for weibo in trending_weibos:
        date = datetime.fromisoformat(weibo.get("create_date_time", ""))
        comment_count = int(weibo["comments_count"])
        if date.year == 2025 and comment_count > 20:
            target_weibos.append({
                "weibo_id": weibo.get("note_id", ""),
                "content": weibo.get("content", ""), 
                "create_date_time": weibo.get("create_date_time", ""),
                "like_count": int(weibo.get("liked_count", 0)),
                "comment_count": int(weibo.get("comments_count", 0)),
                "repost_count": int(weibo.get("shared_count", 0)),
                "ip_location": weibo.get("ip_location", ""), 
                "user_id": weibo.get("user_id", ""), 
                "screen_name": weibo.get("nickname", ""), 
                "gender": weibo.get("gender", ""), 
                "source_keyword": weibo.get("source_keyword", ""),
            })
    return target_weibos

def get_weibo_ids(weibos: List[Dict]) -> List[str]:
    return [weibo.get("weibo_id", "") for weibo in weibos]

In [61]:
target_weibos = clean_weibos(trending_weibos)

weibo_ids = get_weibo_ids(target_weibos)

len(trending_weibos), len(target_weibos), len(weibo_ids)

(8093, 5009, 5009)

In [62]:
weibo_ids = [weibo["weibo_id"] for weibo in target_weibos]

with open("weibo_ids.txt", 'w', encoding='utf-8') as f:
    for weibo_id in weibo_ids:
        f.write(f"{weibo_id}\n")

In [64]:
with open("detail_comments_2026-01-16.json", 'r', encoding='utf-8') as f:
    comments_data = json.load(f)

In [65]:
weibo_id_counter = Counter(comment["note_id"] for comment in comments_data)

weibo_id_counter

Counter({'5195481661572203': 111,
         '5185241507169910': 108,
         '5223108983654695': 106,
         '5195500761911762': 105,
         '5160505140580362': 102,
         '5223110724290198': 102,
         '5195471320518493': 99,
         '5177193694498724': 98,
         '5195496526186227': 98,
         '5177192956301027': 96,
         '5195482867437963': 96,
         '5177198786121230': 92,
         '5159341078351297': 90,
         '5223277607520574': 90,
         '5161049451923609': 88,
         '5223073571932588': 88,
         '5177421089212462': 86,
         '5177195203137760': 84,
         '5177044364693608': 83,
         '5166834933564209': 82,
         '5149216293260201': 82,
         '5166834774704913': 80,
         '5166837553692963': 80,
         '5195482697827612': 80,
         '5137237966328847': 79,
         '5166824817165856': 79,
         '5185193796701419': 78,
         '5159403558014585': 76,
         '5150316995809064': 76,
         '5195512040916643': 76,
    

In [ ]:
import re

def clean_comments(comments: List[Dict]) -> List[Dict]:
    """按时间年份过滤评论，并将二级评论合并至一级评论下

    Args:
        comments (List[Dict]): 原始评论数据

    Returns:
        List[Dict]: 清洗后的评论数据
    """
    comments.sort(key=lambda x: x.get("comment_id", ""))

    target_comments = {}
    for comment in comments:
        comment_id = comment.get("comment_id", "")
        parent_id = comment.get("parent_comment_id", "")
        date = datetime.fromisoformat(comment.get("create_date_time", ""))
        content = clean_text(comment.get("content", ""))
        if date.year == 2025 and len(content) > 0:
            # 一级评论
            if comment_id == parent_id:
                target_comments[comment_id] = {
                    "comment_id": comment_id,
                    "weibo_id": comment.get("note_id", ""),
                    "content": content,
                    "create_date_time": comment.get("create_date_time", ""),
                    "like_count": int(comment.get("comment_like_count", 0)),
                    "sub_comment_count": int(comment.get("sub_comment_count", 0)),
                    "sub_comments": [], 
                    "ip_location": comment.get("ip_location", ""),
                    "user_id": comment.get("user_id", ""),
                    "screen_name": comment.get("nickname", ""),
                    "gender": comment.get("gender", ""),
                }
            # 二级评论
            else:
                if parent_id in target_comments:
                    target_comments[parent_id]["sub_comments"].append({
                        "comment_id": comment_id,
                        "weibo_id": comment.get("note_id", ""),
                        "content": content,
                        "create_date_time": comment.get("create_date_time", ""),
                        "like_count": int(comment.get("comment_like_count", 0)),
                        "ip_location": comment.get("ip_location", ""),
                        "user_id": comment.get("user_id", ""),
                        "screen_name": comment.get("nickname", ""),
                        "gender": comment.get("gender", ""),
                    })
                else:
                    print(f"找不到二级评论 {comment_id} 的父评论 {parent_id}，已跳过")

    return list(target_comments.values())



def clean_text(text: str) -> str:
    """清洗评论文本，去除无效内容

    Args:
        text (str): 原始评论文本

    Returns:
        str: 清洗后的评论文本
    """
    # 去除“图片评论”“评论配图”“转发微博”等文字
    text = re.sub(r'图片评论|评论配图|转发微博|转发了|网页链接', '', text)

    # 去除多余空格
    text = re.sub(r'[\s]+', ' ', text)

    return text.strip()

In [72]:
cleaned_comments = clean_comments(comments_data)

len(comments_data), len(cleaned_comments)

找不到二级评论 5131569755062559 的父评论 5131547592625331，已跳过
找不到二级评论 5150588279196299 的父评论 5150572554486361，已跳过
找不到二级评论 5150588279196299 的父评论 5150572554486361，已跳过
找不到二级评论 5160676059251322 的父评论 5160527346010324，已跳过
找不到二级评论 5160676059251322 的父评论 5160527346010324，已跳过
找不到二级评论 5160679749455297 的父评论 5160527346010324，已跳过
找不到二级评论 5160679749455297 的父评论 5160527346010324，已跳过
找不到二级评论 5166849135216519 的父评论 5166847017618781，已跳过
找不到二级评论 5166849135216519 的父评论 5166847017618781，已跳过
找不到二级评论 5173218555728456 的父评论 5173116812659217，已跳过
找不到二级评论 5176107208808323 的父评论 5173116812659217，已跳过
找不到二级评论 5176786284972935 的父评论 5176762256591902，已跳过
找不到二级评论 5176786284972935 的父评论 5176762256591902，已跳过
找不到二级评论 5193816548050540 的父评论 5193816451842581，已跳过
找不到二级评论 5193821039624857 的父评论 5193816451842581，已跳过
找不到二级评论 5193825545094192 的父评论 5193825158959690，已跳过
找不到二级评论 5193857121650273 的父评论 5193853930834247，已跳过
找不到二级评论 5199091560875080 的父评论 5198902757692045，已跳过
找不到二级评论 5223443416744240 的父评论 5223271517127177，已跳过


(14569, 6555)

In [68]:
for comment in cleaned_comments:
    if comment["sub_comment_count"] != 0:
        print(comment)
        break

{'comment_id': '5119080710800368', 'weibo_id': '5119078547588275', 'content': '', 'create_date_time': '2025-01-04 13:10:12+08:00', 'like_count': 35, 'sub_comment_count': 9, 'sub_comments': [], 'ip_location': '澳大利亚', 'user_id': '7336877965', 'screen_name': 'wode公主殿下', 'gender': 'f'}


In [70]:
cleaned_comments

[{'comment_id': '5119080710800368',
  'weibo_id': '5119078547588275',
  'content': '',
  'create_date_time': '2025-01-04 13:10:12+08:00',
  'like_count': 35,
  'sub_comment_count': 9,
  'sub_comments': [],
  'ip_location': '澳大利亚',
  'user_id': '7336877965',
  'screen_name': 'wode公主殿下',
  'gender': 'f'},
 {'comment_id': '5119083840279093',
  'weibo_id': '5119078547588275',
  'content': '马龙持续书写新的传奇',
  'create_date_time': '2025-01-04 13:22:38+08:00',
  'like_count': 293,
  'sub_comment_count': 0,
  'sub_comments': [],
  'ip_location': '上海',
  'user_id': '7910039613',
  'screen_name': '幸运鹅-黑化版',
  'gender': 'f'},
 {'comment_id': '5119084048154967',
  'weibo_id': '5119078547588275',
  'content': '马龙，只要心怀热爱永远是当打之年！',
  'create_date_time': '2025-01-04 13:23:27+08:00',
  'like_count': 210,
  'sub_comment_count': 0,
  'sub_comments': [],
  'ip_location': '湖北',
  'user_id': '1909111994',
  'screen_name': '在_寶呗',
  'gender': 'f'},
 {'comment_id': '5119084364566333',
  'weibo_id': '51190785475882